# Cargar ubicación de paradas (Intendencia de Montevideo)
Descarga la capa de paradas de ómnibus de los datos abiertos de la IM,
convierte las coordenadas a latitud/longitud y guarda la tabla `paradas` en Databricks.

**Uso:** Run all. Al final te dice cuántas paradas de la tabla de Ariana encontraron ubicación.

In [ ]:
%pip install pyshp pyproj --quiet

In [ ]:
dbutils.library.restartPython()

In [ ]:
# --- Configuración ---
TABLA_DESTINO = "workspace.movilidad_silver.paradas"
TABLA_OFERTA  = "workspace.movilidad_gold.oferta_parada_hora"

# Plan B: si la descarga automática falla, bajá el .zip de paradas a mano desde
# el catálogo de datos abiertos de la IM, subilo a un Volume y poné la ruta acá.
# Ej: "/Volumes/workspace/movilidad_bronze/localizacion_paradas/v_uptu_paradas.zip"
RUTA_ZIP_MANUAL = ""

# Si sabés la URL exacta del archivo de paradas (por ejemplo la que usa tu script
# de descarga), ponela acá y se usa primero.
URL_PARADAS = ""

# Catálogos de datos abiertos (los mismos de donde salen horarios y viajes)
CATALOGOS_CKAN = ["https://catalogodatos.gub.uy", "https://ckan.montevideo.gub.uy"]
DATASET_PARADAS = "transporte-colectivo-paradas-puntos-de-control-y-recorridos-de-omnibus"

# Respaldo: servidores de mapas de la IM (desde algunos Databricks no responden)
_ZIP_IM = "http://intgis.montevideo.gub.uy/sit/php/common/datos/generar_zip2.php?nom_tab=v_uptu_paradas&tipo=gis"
FUENTES_RESPALDO = [
    "https://intgis.montevideo.gub.uy/sit/php/common/datos/generar_zip2.php?nom_tab=v_uptu_paradas&tipo=gis",
    "https://geoweb.montevideo.gub.uy/geoserver/imm/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=imm:v_uptu_paradas&outputFormat=application/json&srsName=EPSG:4326",
    # Copia guardada del mismo zip en el Internet Archive (accesible desde fuera de Uruguay)
    "https://web.archive.org/web/2026id_/" + _ZIP_IM,
    "https://web.archive.org/web/2025id_/" + _ZIP_IM,
    "https://web.archive.org/web/2024id_/" + _ZIP_IM,
]

In [ ]:
import io, json, zipfile, requests
import pandas as pd
import shapefile
from pyproj import CRS, Transformer

def urls_del_catalogo():
    """Pregunta al catálogo qué archivos tiene el dataset y devuelve primero los de paradas."""
    urls = []
    for base in CATALOGOS_CKAN:
        try:
            r = requests.get(f"{base}/api/3/action/package_show", params={"id": DATASET_PARADAS}, timeout=60)
            recursos = r.json()["result"]["resources"]
            print(f"Catálogo {base}: {len(recursos)} archivos en el dataset")
            for rec in recursos:
                texto = f"{rec.get('name','')} {rec.get('description','')} {rec.get('url','')}".lower()
                print("  -", rec.get("name"), "|", rec.get("format"), "|", rec.get("url"))
                if "parada" in texto:
                    urls.append(rec["url"])
        except Exception as e:
            print(f"Catálogo {base}: no respondió ({str(e)[:80]})")
    return urls

def fuentes():
    return ([URL_PARADAS] if URL_PARADAS else []) + urls_del_catalogo() + FUENTES_RESPALDO

CARPETA_SUBIDAS = "/Volumes/workspace/movilidad_bronze/localizacion_paradas"

def buscar_zip_subido():
    """Busca un .zip subido a mano en el Volume localizacion_paradas."""
    import os
    try:
        zips = [f for f in os.listdir(CARPETA_SUBIDAS) if f.lower().endswith(".zip")]
    except Exception:
        return None
    if not zips:
        return None
    preferido = next((f for f in zips if "parada" in f.lower()), zips[0])
    return f"{CARPETA_SUBIDAS}/{preferido}"

def descargar():
    ruta = RUTA_ZIP_MANUAL or buscar_zip_subido()
    if ruta:
        print("Usando archivo subido:", ruta)
        with open(ruta, "rb") as f:
            return f.read(), ruta
    for url in fuentes():
        try:
            print("Probando:", url[:90], "…")
            r = requests.get(url, timeout=120, headers={"User-Agent": "Mozilla/5.0"})
            util = r.content[:2] == b"PK" or r.content.lstrip()[:1] == b"{"
            if r.status_code == 200 and len(r.content) > 1000 and util:
                print("  OK,", round(len(r.content) / 1e6, 2), "MB")
                return r.content, url
            print("  respuesta", r.status_code, "-", len(r.content), "bytes")
        except Exception as e:
            print("  falló:", str(e)[:120])
    raise RuntimeError("No se pudo descargar automáticamente. Usá RUTA_ZIP_MANUAL (ver arriba).")

contenido, origen = descargar()

In [ ]:
def col(cols, *claves):
    """Encuentra una columna por parte del nombre, sin importar mayúsculas."""
    for c in cols:
        if any(k in c.lower() for k in claves):
            return c
    return None

filas = []

if contenido[:2] == b"PK":                     # ---- shapefile en zip
    z = zipfile.ZipFile(io.BytesIO(contenido))
    nombres = z.namelist()
    base = next(n[:-4] for n in nombres if n.lower().endswith(".shp"))
    def leer(ext):
        n = next((x for x in nombres if x.lower() == (base + ext).lower()), None)
        return io.BytesIO(z.read(n)) if n else None
    sf = shapefile.Reader(shp=leer(".shp"), dbf=leer(".dbf"), shx=leer(".shx"), encoding="latin-1")
    prj = leer(".prj")
    crs_origen = CRS.from_wkt(prj.read().decode("latin-1")) if prj else CRS.from_epsg(32721)
    print("Sistema de coordenadas de origen:", crs_origen.name)
    campos = [f[0] for f in sf.fields[1:]]
    for rec, shp in zip(sf.records(), sf.shapes()):
        if not shp.points:
            continue
        d = dict(zip(campos, rec))
        d["_x"], d["_y"] = shp.points[0]
        filas.append(d)
    df = pd.DataFrame(filas)
    t = Transformer.from_crs(crs_origen, CRS.from_epsg(4326), always_xy=True)
    df["lon"], df["lat"] = t.transform(df["_x"].values, df["_y"].values)
elif contenido.lstrip()[:1] == b"{" and b'"elements"' in contenido[:2000]:   # ---- OpenStreetMap
    print("AVISO: se usaron paradas de OpenStreetMap, no la capa oficial de la IM.")
    for el in json.loads(contenido)["elements"]:
        tags = el.get("tags", {})
        filas.append({"cod_parada": tags.get("ref") or tags.get("ref:IM") or tags.get("ref:stm"),
                      "calle": tags.get("name"), "lat": el["lat"], "lon": el["lon"]})
    df = pd.DataFrame(filas)
elif contenido.lstrip()[:1] in (b"{", b"["):     # ---- GeoJSON
    gj = json.loads(contenido)
    for ft in gj["features"]:
        d = dict(ft["properties"])
        d["lon"], d["lat"] = ft["geometry"]["coordinates"][:2]
        filas.append(d)
    df = pd.DataFrame(filas)
else:                                           # ---- CSV con columnas X/Y o lat/lon
    texto = contenido.decode("utf-8", errors="replace")
    sep = ";" if texto.splitlines()[0].count(";") > texto.splitlines()[0].count(",") else ","
    df = pd.read_csv(io.StringIO(texto), sep=sep)
    c_lat, c_lon = col(df.columns, "lat"), col(df.columns, "lon", "lng")
    if c_lat and c_lon:
        df["lat"], df["lon"] = pd.to_numeric(df[c_lat], errors="coerce"), pd.to_numeric(df[c_lon], errors="coerce")
    else:
        c_x = next(c for c in df.columns if c.lower() in ("x", "coord_x", "xcoord", "punto_x"))
        c_y = next(c for c in df.columns if c.lower() in ("y", "coord_y", "ycoord", "punto_y"))
        t = Transformer.from_crs(CRS.from_epsg(32721), CRS.from_epsg(4326), always_xy=True)
        df["lon"], df["lat"] = t.transform(pd.to_numeric(df[c_x]).values, pd.to_numeric(df[c_y]).values)

print(len(df), "registros. Columnas:", list(df.columns))

In [ ]:
c_cod     = col(df.columns, "ubic", "cod_parada", "codigo_parada", "parada")
c_calle   = col(df.columns, "calle")
c_esquina = col(df.columns, "esquina")
c_linea   = col(df.columns, "desc_linea", "linea")
assert c_cod, f"No encontré la columna del código de parada en {list(df.columns)}"
print("Código de parada:", c_cod, "| calle:", c_calle, "| esquina:", c_esquina)

df["codigo_parada"] = pd.to_numeric(df[c_cod], errors="coerce")
df = df.dropna(subset=["codigo_parada", "lat", "lon"])
assert len(df) > 0, ("El archivo no trae códigos de parada que se puedan usar "
                     "(pasa si salió de OpenStreetMap). Avisale a Claude qué fuente se usó.")
df["codigo_parada"] = df["codigo_parada"].astype("int64")

# La capa repite cada parada una vez por línea: quedarse con una fila por parada
agg = {"lat": "mean", "lon": "mean"}
if c_calle:   agg[c_calle] = "first"
if c_esquina: agg[c_esquina] = "first"
paradas = df.groupby("codigo_parada", as_index=False).agg(agg)
if c_linea:
    lineas = df.groupby("codigo_parada")[c_linea].apply(lambda s: ", ".join(sorted(set(map(str, s)))))
    paradas["lineas"] = paradas["codigo_parada"].map(lineas)
paradas = paradas.rename(columns={c_calle: "calle", c_esquina: "esquina"} if c_calle else {})

# Control: Montevideo está cerca de lat -34.9, lon -56.2
dentro = paradas["lat"].between(-35.1, -34.5) & paradas["lon"].between(-56.7, -55.7)
print(f"{len(paradas)} paradas únicas, {dentro.mean():.1%} dentro de la zona de Montevideo")
assert dentro.mean() > 0.9, "Las coordenadas no caen en Montevideo: la conversión salió mal."
display(paradas.head(10))

In [ ]:
sdf = spark.createDataFrame(paradas)
sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_DESTINO)
print("Guardada en", TABLA_DESTINO)

In [ ]:
# Cuántas paradas de la tabla de oferta encontraron su ubicación
display(spark.sql(f"""
SELECT COUNT(DISTINCT o.codigo_parada)                                   AS paradas_con_horario,
       COUNT(DISTINCT CASE WHEN p.codigo_parada IS NOT NULL THEN o.codigo_parada END) AS con_ubicacion,
       ROUND(100 * COUNT(DISTINCT CASE WHEN p.codigo_parada IS NOT NULL THEN o.codigo_parada END)
             / COUNT(DISTINCT o.codigo_parada), 1)                        AS pct_con_ubicacion
FROM {TABLA_OFERTA} o
LEFT JOIN {TABLA_DESTINO} p ON o.codigo_parada = p.codigo_parada
"""))